Ways to read data from table/view:
1) spark.table(tablename)
2) spark.table.read(tablename).option(if any) //More Flexible
2) spark.sql(query)

Ways to drop dupes:
1) df.dropDuplicates()
2) df.distinct()

Ways to filter data:
1) df.filter('col is not null') 
2) df.filter(df.col.isNotNull()) //can use WHERE instead of FILTER as well

Select data:
1) df.select(cols/list)


In [0]:
df = spark.table('gizmobox.bronze.py_customers')
display(df)

In [0]:
df = spark.table('gizmobox.bronze.py_customers')
df_filtered = df.filter(df.customer_id.isNotNull())
display(df_filtered)

In [0]:
df_distinct = df_filtered.dropDuplicates()
display(df_distinct)

In [0]:
from pyspark.sql import functions as F

df_max_ts = df_distinct.groupBy('customer_id').agg(F.max('created_timestamp').alias('max_created_timestamp'))

display(df_max_ts)

In [0]:
df_distinct_customer = df_distinct.join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) & (df_distinct.created_timestamp == df_max_ts.max_created_timestamp), "inner").select(df_distinct["*"])

display(df_distinct_customer)

In [0]:
df_casted_customer = df_distinct_customer.select(
    df_distinct_customer.created_timestamp.cast("timestamp"),
    df_distinct_customer.customer_id,
    df_distinct_customer.customer_name,
    df_distinct_customer.date_of_birth.cast("date"),
    df_distinct_customer.email,
    df_distinct_customer.member_since.cast("date"),
    df_distinct_customer.telephone
)

display(df_casted_customer)

In [0]:
df_casted_customer.writeTo('gizmobox.silver.py_customers').createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_customers;